In [12]:
import pandas as pd
import glob

# ============================================================
# SANITY CHECK - CIC-IoT2023 dataset
# Mục tiêu: verify dataset load đúng, đúng schema Neto 2023
# ============================================================

# --- 1. Liệt kê toàn bộ file part trong MERGED_CSV ---
# CIC-IoT2023 sau giải nén có ~169 file part-*.csv (hoặc Merged*.csv tùy phiên bản)
# Dùng glob để đếm và lấy danh sách, KHÔNG hard-code 1 file để tránh nhầm lẫn
files = sorted(glob.glob("../data/raw/archive/CICIOT23/train/train.csv"))
print(f"Số file CSV tìm thấy: {len(files)}")
print(f"File đầu: {files[0]}")
print(f"File cuối: {files[-1]}")
# Kỳ vọng: ~169 file (hoặc ít hơn nếu bản của bạn đã được merge sẵn thành ít file lớn)

# --- 2. Load 1 file mẫu để xem schema (KHÔNG load hết 13GB) ---
# Chỉ đọc file đầu tiên, đủ để verify cấu trúc cột
df_sample = pd.read_csv(files[0])

# --- 3. Shape: số dòng × số cột ---
# Kỳ vọng số cột = 47 (46 feature + 1 Label)
# Số dòng mỗi file ~280k (tổng 46,68M chia đều 169 file)
print(f"\nShape file mẫu: {df_sample.shape}")
print(f"  → {df_sample.shape[0]:,} dòng × {df_sample.shape[1]} cột")

# --- 4. Verify số cột = 47 (nếu không đúng là dataset lỗi hoặc bản khác) ---
assert df_sample.shape[1] == 47, (
    f"Số cột phải là 47 (46 feature + 1 Label), nhưng có {df_sample.shape[1]}. "
    f"Kiểm tra lại nguồn download từ UNB CIC."
)
print(f"  ✓ Số cột = 47 (khớp với 46 feature + 1 Label của Neto 2023)")

# --- 5. In danh sách tên cột ---
# Dùng để verify từng cột khớp với bảng trên trang UNB CIC
# 46 feature đầu: flow_duration, Header_Length, Protocol type, ..., Weight
# Cột cuối cùng: Label (hoặc label - tùy phiên bản, đề cương dùng 'Label')
print(f"\nDanh sách 47 cột:")
for i, col in enumerate(df_sample.columns.tolist(), start=1):
    # Đánh dấu cột Label để dễ nhìn
    marker = "  ← LABEL" if col.lower() == 'label' else ""
    print(f"  {i:2d}. {col}{marker}")

# --- 6. Xác nhận cột cuối là Label (để pipeline SAE biết cột nào là target) ---
label_col = df_sample.columns[-1]
print(f"\nTên cột label: '{label_col}'")
# Lưu ý: một số mirror dùng 'label' lowercase, một số dùng 'Label' capitalized
# Code sau phải xử lý nhất quán - đề cương §6.1 dùng 'Label'

# --- 7. Đếm class trong file mẫu ---
# CẢNH BÁO: đây chỉ là phân phối trong 1 file (~280k dòng),
# KHÔNG phải phân phối toàn dataset 46,68M dòng.
# Chỉ dùng để sanity check các class xuất hiện có tên hợp lệ không.
print(f"\nPhân phối class trong file mẫu này ({df_sample.shape[0]:,} dòng):")
value_counts = df_sample[label_col].value_counts()
print(value_counts)
print(f"\n  → Số class xuất hiện trong file mẫu: {len(value_counts)}")
# Kỳ vọng: <= 34 class (một file có thể không chứa đủ tất cả 34 class,
# vì dữ liệu đã shuffle nhưng 1 file ~280k chưa chắc đại diện toàn bộ)

# --- 8. Kiểm tra kiểu dữ liệu ---
# Kỳ vọng: 46 cột numeric (int64 hoặc float64) + 1 cột object (Label là string)
print(f"\nPhân bố kiểu dữ liệu:")
print(df_sample.dtypes.value_counts())
n_numeric = df_sample.select_dtypes(include=['number']).shape[1]
n_object = df_sample.select_dtypes(include=['object']).shape[1]
print(f"  → {n_numeric} cột numeric (feature), {n_object} cột object (label)")
assert n_numeric == 46, f"Phải có 46 cột numeric, nhưng có {n_numeric}"
assert n_object == 1, f"Phải có 1 cột object (Label), nhưng có {n_object}"
print(f"  ✓ Schema đúng: 46 feature numeric + 1 Label object")

# --- 9. Kiểm tra NaN/Inf sơ bộ (sẽ xử lý kỹ ở preprocessing) ---
# Thường Rate/Srate có Inf do flow_duration = 0 → chia cho 0
# Đề cương §6.1 nói: "thay NaN/Inf bằng 0 sau khi log các flow bị lỗi"
import numpy as np
n_nan = df_sample.isna().sum().sum()
n_inf = np.isinf(df_sample.select_dtypes(include=['number'])).sum().sum()
print(f"\nData quality check:")
print(f"  NaN values: {n_nan:,}")
print(f"  Inf values: {n_inf:,}")
if n_inf > 0:
    # Chỉ ra cột nào bị Inf để biết đường xử lý ở preprocessing
    inf_by_col = np.isinf(df_sample.select_dtypes(include=['number'])).sum()
    print(f"  Cột bị Inf:")
    print(inf_by_col[inf_by_col > 0])

Số file CSV tìm thấy: 1
File đầu: ../data/raw/archive/CICIOT23/train/train.csv
File cuối: ../data/raw/archive/CICIOT23/train/train.csv

Shape file mẫu: (5491971, 47)
  → 5,491,971 dòng × 47 cột
  ✓ Số cột = 47 (khớp với 46 feature + 1 Label của Neto 2023)

Danh sách 47 cột:
   1. flow_duration
   2. Header_Length
   3. Protocol Type
   4. Duration
   5. Rate
   6. Srate
   7. Drate
   8. fin_flag_number
   9. syn_flag_number
  10. rst_flag_number
  11. psh_flag_number
  12. ack_flag_number
  13. ece_flag_number
  14. cwr_flag_number
  15. ack_count
  16. syn_count
  17. fin_count
  18. urg_count
  19. rst_count
  20. HTTP
  21. HTTPS
  22. DNS
  23. Telnet
  24. SMTP
  25. SSH
  26. IRC
  27. TCP
  28. UDP
  29. DHCP
  30. ARP
  31. ICMP
  32. IPv
  33. LLC
  34. Tot sum
  35. Min
  36. Max
  37. AVG
  38. Std
  39. Tot size
  40. IAT
  41. Number
  42. Magnitue
  43. Radius
  44. Covariance
  45. Variance
  46. Weight
  47. label  ← LABEL

Tên cột label: 'label'

Phân phối class trong

/tmp/ipykernel_50847/3629399327.py:67: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  n_object = df_sample.select_dtypes(include=['object']).shape[1]



Data quality check:
  NaN values: 0
  Inf values: 0


In [11]:
import pandas as pd
import glob

# Đọc 1 file thử để xem schema
df_sample = pd.read_csv("../data/raw/CSV/Backdoor_Malware/Backdoor_Malware.pcap.csv")
print(df_sample.shape)
print(df_sample.columns.tolist())
# print(df_sample['Label'].value_counts())

(3218, 39)
['Header_Length', 'Protocol Type', 'Time_To_Live', 'Rate', 'fin_flag_number', 'syn_flag_number', 'rst_flag_number', 'psh_flag_number', 'ack_flag_number', 'ece_flag_number', 'cwr_flag_number', 'ack_count', 'syn_count', 'fin_count', 'rst_count', 'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP', 'SSH', 'IRC', 'TCP', 'UDP', 'DHCP', 'ARP', 'ICMP', 'IGMP', 'IPv', 'LLC', 'Tot sum', 'Min', 'Max', 'AVG', 'Std', 'Tot size', 'IAT', 'Number', 'Variance']


In [14]:
import glob, pandas as pd
from collections import Counter

files = sorted(glob.glob('../data/raw/archive/CICIOT23/train/train.csv'))
print(f"Số file: {len(files)}")  # kỳ vọng 63

label_counts = Counter()
total_rows = 0

for f in files:
    df = pd.read_csv(f, usecols=['label'])
    label_counts.update(df['label'].value_counts().to_dict())
    total_rows += len(df)

print(f"Tổng rows: {total_rows:,}")
print(pd.Series(label_counts).sort_values(ascending=False).to_string())

Số file: 1
Tổng rows: 5,491,971
DDoS-ICMP_Flood            848088
DDoS-UDP_Flood             637558
DDoS-TCP_Flood             528499
DDoS-PSHACK_Flood          481254
DDoS-SYN_Flood             478653
DDoS-RSTFINFlood           475441
DDoS-SynonymousIP_Flood    422083
DoS-UDP_Flood              390422
DoS-TCP_Flood              314174
DoS-SYN_Flood              237573
BenignTraffic              129538
Mirai-greeth_flood         116133
Mirai-udpplain             104814
Mirai-greip_flood           88821
DDoS-ICMP_Fragmentation     53046
MITM-ArpSpoofing            36316
DDoS-UDP_Fragmentation      34169
DDoS-ACK_Fragmentation      33581
DNS_Spoofing                21214
Recon-HostDiscovery         15737
Recon-OSScan                11587
Recon-PortScan               9648
DoS-HTTP_Flood               8487
VulnerabilityScan            4396
DDoS-HTTP_Flood              3371
DDoS-SlowLoris               2757
DictionaryBruteForce         1541
BrowserHijacking              665
CommandInjection

In [1]:
import pandas as pd
import numpy as np
import glob, os, pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ── Paths ──────────────────────────────────────────────────────────────
RAW_DIR = '../data/raw/MERGED_CSV/'
OUT_DIR = '../data/processed/'
os.makedirs(OUT_DIR, exist_ok=True)

# ── Load 63 file bằng pandas concat ───────────────────────────────────
files = sorted(glob.glob(RAW_DIR + 'Merged*.csv'))
print(f"Số file: {len(files)}")  # phải ra 63

print("Loading... (~5–10 phút, peak RAM ~14 GB)")
df = pd.concat(
    [pd.read_csv(f) for f in files],
    ignore_index=True
)
print(f"Shape: {df.shape}")  # kỳ vọng (~46.6M, 47)
print(f"Columns: {df.columns.tolist()}")

# # ── Split 80/10/10 stratified ─────────────────────────────────────────
# X = df.drop(columns=['Label'])
# y = df['Label']

# X_tr, X_tmp, y_tr, y_tmp = train_test_split(
#     X, y, test_size=0.2, stratify=y, random_state=42)
# X_val, X_te, y_val, y_te = train_test_split(
#     X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=42)

# print(f"Train: {len(X_tr):,} | Val: {len(X_val):,} | Test: {len(X_te):,}")
# del df, X, y, X_tmp, y_tmp  # giải phóng RAM

# # ── Scale (fit trên benign train only) ────────────────────────────────
# scaler = StandardScaler()
# benign_mask = (y_tr == 'BenignTraffic')   # ← kiểm tra tên nhãn thực tế
# scaler.fit(X_tr[benign_mask])

# def scale_and_clean(X_arr):
#     out = scaler.transform(X_arr)
#     out[~np.isfinite(out)] = 0.0
#     return out

# # ── Export parquet ─────────────────────────────────────────────────────
# feature_cols = X_tr.columns.tolist()
# for split_name, X_split, y_split in [
#     ('train', X_tr,  y_tr),
#     ('val',   X_val, y_val),
#     ('test',  X_te,  y_te),
# ]:
#     scaled = scale_and_clean(X_split)
#     out_df = pd.DataFrame(scaled, columns=feature_cols)
#     out_df['Label'] = y_split.values
#     out_df.to_parquet(f'{OUT_DIR}{split_name}.parquet', index=False)
#     print(f"Saved {split_name}.parquet — {len(out_df):,} rows")

# with open(OUT_DIR + 'scaler_benign.pkl', 'wb') as f:
#     pickle.dump(scaler, f)
# print("Done.")

Số file: 63
Loading... (~5–10 phút, peak RAM ~14 GB)
Shape: (45019243, 40)
Columns: ['Header_Length', 'Protocol Type', 'Time_To_Live', 'Rate', 'fin_flag_number', 'syn_flag_number', 'rst_flag_number', 'psh_flag_number', 'ack_flag_number', 'ece_flag_number', 'cwr_flag_number', 'ack_count', 'syn_count', 'fin_count', 'rst_count', 'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP', 'SSH', 'IRC', 'TCP', 'UDP', 'DHCP', 'ARP', 'ICMP', 'IGMP', 'IPv', 'LLC', 'Tot sum', 'Min', 'Max', 'AVG', 'Std', 'Tot size', 'IAT', 'Number', 'Variance', 'Label']


In [2]:
# Chẩn đoán NaN
print(f"Total NaN in Label: {df['Label'].isna().sum():,}")
print(f"Total NaN in features: {df.drop(columns=['Label']).isna().sum().sum():,}")
print(f"Total Inf in features: {np.isinf(df.drop(columns=['Label']).select_dtypes(include=[np.number])).sum().sum():,}")
print(f"\nLabel value counts (top 10):")
print(df['Label'].value_counts(dropna=False).head(10))

Total NaN in Label: 9
Total NaN in features: 1,489
Total Inf in features: 991

Label value counts (top 10):
Label
DDOS-ICMP_FLOOD            6893259
DDOS-UDP_FLOOD             5181027
DDOS-TCP_FLOOD             4306086
DDOS-PSHACK_FLOOD          3920372
DDOS-SYN_FLOOD             3886130
DDOS-RSTFINFLOOD           3872808
DDOS-SYNONYMOUSIP_FLOOD    3445659
DOS-UDP_FLOOD              3177323
DOS-TCP_FLOOD              2558256
DOS-SYN_FLOOD              1942176
Name: count, dtype: int64


In [3]:
# Tìm label benign chính xác
print([l for l in df['Label'].unique() if 'BENIGN' in str(l).upper()])
print(f"Tổng số class: {df['Label'].nunique()}")

['BENIGN']
Tổng số class: 34


In [2]:
import numpy as np
import pandas as pd
import pickle, os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

OUT_DIR = '../data/processed/'
os.makedirs(OUT_DIR, exist_ok=True)
BENIGN_LABEL = 'BENIGN'

# ── Clean NaN/Inf ──────────────────────────────────────────────────────
n_before = len(df)
df = df.dropna(subset=['Label'])
print(f"Dropped {n_before - len(df):,} rows với Label NaN")

feature_cols = [c for c in df.columns if c != 'Label']
df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
print(f"Shape sau clean: {df.shape}")

# ── Split 80/10/10 stratified ─────────────────────────────────────────
X = df.drop(columns=['Label'])
y = df['Label']

X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
X_val, X_te, y_val, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=42)

print(f"Train: {len(X_tr):,} | Val: {len(X_val):,} | Test: {len(X_te):,}")
del df, X, y, X_tmp, y_tmp

# ── Scale (fit trên benign train only) ────────────────────────────────
scaler = StandardScaler()
benign_mask = (y_tr == BENIGN_LABEL)
print(f"Benign trong train: {benign_mask.sum():,}")
scaler.fit(X_tr[benign_mask])

def scale(X_arr):
    out = scaler.transform(X_arr)
    out[~np.isfinite(out)] = 0.0
    return out

# ── Export parquet ─────────────────────────────────────────────────────
for split_name, X_split, y_split in [
    ('train', X_tr,  y_tr),
    ('val',   X_val, y_val),
    ('test',  X_te,  y_te),
]:
    scaled = scale(X_split)
    out_df = pd.DataFrame(scaled, columns=feature_cols)
    out_df['Label'] = y_split.values
    out_df.to_parquet(f'{OUT_DIR}{split_name}.parquet', index=False)
    print(f"Saved {split_name}.parquet — {len(out_df):,} rows")

with open(OUT_DIR + 'scaler_benign.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("Done.")

Dropped 9 rows với Label NaN
Shape sau clean: (45019234, 40)
Train: 36,015,387 | Val: 4,501,923 | Test: 4,501,924
Benign trong train: 841,098
Saved train.parquet — 36,015,387 rows
Saved val.parquet — 4,501,923 rows
Saved test.parquet — 4,501,924 rows
Done.


In [4]:
# Kiểm tra nhanh 3 parquet trước khi next step
import pandas as pd

for name in ['train', 'val', 'test']:
    df = pd.read_parquet(f'../data/processed/{name}.parquet')
    print(f"\n{name}: {df.shape}")
    print(f"  Class count: {df['Label'].nunique()}")
    print(f"  Benign: {(df['Label']=='BENIGN').sum():,}")
    print(f"  Top 3 attacks: {df['Label'].value_counts().head(3).to_dict()}")
    print(f"  NaN check: {df.isna().sum().sum()}")
    print(f"  Inf check: {df.select_dtypes(include='number').apply(lambda x: (x==float('inf')).sum()).sum()}")


train: (36015387, 40)
  Class count: 34
  Benign: 841,098
  Top 3 attacks: {'DDOS-ICMP_FLOOD': 5514607, 'DDOS-UDP_FLOOD': 4144822, 'DDOS-TCP_FLOOD': 3444869}
  NaN check: 0
  Inf check: 0

val: (4501923, 40)
  Class count: 34
  Benign: 105,137
  Top 3 attacks: {'DDOS-ICMP_FLOOD': 689326, 'DDOS-UDP_FLOOD': 518102, 'DDOS-TCP_FLOOD': 430608}
  NaN check: 0
  Inf check: 0

test: (4501924, 40)
  Class count: 34
  Benign: 105,138
  Top 3 attacks: {'DDOS-ICMP_FLOOD': 689326, 'DDOS-UDP_FLOOD': 518103, 'DDOS-TCP_FLOOD': 430609}
  NaN check: 0
  Inf check: 0


In [6]:
import pandas as pd
for name in ['train', 'val', 'test']:
    df = pd.read_parquet(f'../data/processed/{name}.parquet')
    print(f"\n{name} — 3 lớp nhỏ nhất:")
    print(df['Label'].value_counts().tail(3))


train — 3 lớp nhỏ nhất:
Label
BACKDOOR_MALWARE    2462
RECON-PINGSWEEP     1729
UPLOADING_ATTACK     957
Name: count, dtype: int64

val — 3 lớp nhỏ nhất:
Label
BACKDOOR_MALWARE    308
RECON-PINGSWEEP     216
UPLOADING_ATTACK    120
Name: count, dtype: int64

test — 3 lớp nhỏ nhất:
Label
BACKDOOR_MALWARE    308
RECON-PINGSWEEP     216
UPLOADING_ATTACK    119
Name: count, dtype: int64
